# 파인튜닝(Fine-tuning) 가이드

---

## 1. 개요 및 환경 설정

### 1.1 파인튜닝이란?

- **파인튜닝(Fine-tuning)** 은 사전 학습된 모델을 특정 도메인이나 작업에 맞게 추가 학습시키는 과정

    - 도메인 특화 지식 습득 (예: ETF 투자 전문 지식)
    - 응답 스타일 및 포맷 커스터마이징
    - 일반 모델보다 높은 정확도
    - 비용 효율적 (처음부터 학습하는 것보다)

- **LLM vs 임베딩 모델**

    | 특징 | LLM 파인튜닝 | 임베딩 모델 파인튜닝 |
    |------|-------------|-------------------|
    | 목적 | 텍스트 생성, 질의응답 | 의미적 유사도 계산, 검색 |
    | 출력 | 자연어 텍스트 | 벡터 임베딩 |
    | 주요 사용처 | 챗봇, 요약, 번역 | RAG, 시맨틱 검색, 추천 |
    | 데이터 형식 | Q&A 쌍, 대화 | 유사 문장 쌍, triplets |


### 1.2 환경 설정

- **필수 라이브러리 설치**

    ```bash
    # Unsloth (LLM 파인튜닝용)
    pip install unsloth / uv pip install unsloth

    # Sentence Transformers (임베딩 모델용)
    pip install -U "sentence-transformers>=3.0"

    # 공통 라이브러리
    pip install datasets accelerate torch pandas
    pip install python-dotenv  # 환경변수 관리용
    ```

- **하드웨어 요구사항**

- **LLM 파인튜닝 (Unsloth)**
    - GPU: NVIDIA GPU 16GB+ VRAM 권장
    - 4-bit 양자화로 메모리 4배 절감 가능
    - Google Colab (무료 T4), Runpod, Lambda Labs 등 활용 가능

- **임베딩 모델 파인튜닝**
    - GPU: 8GB+ VRAM
    - CPU에서도 가능하지만 느림
    - Colab 무료 티어로도 충분

- **Hugging Face 토큰 설정**

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()

# 또는 직접 입력
if "HUGGINGFACE_TOKEN" not in os.environ:
    os.environ["HUGGINGFACE_TOKEN"] = getpass("Hugging Face Token: ")
    
if "OPENAI_API_KEY" not in os.environ:
    use_openai = input("OpenAI API 사용? (y/n): ")
    if use_openai.lower() == 'y':
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# Hugging Face 로그인
from huggingface_hub import login
login(token=os.environ["HUGGINGFACE_TOKEN"], add_to_git_credential=True)

print("✅ 환경 설정 완료!")

---

## 2. 데이터 수집 및 전처리

- 한국 ETF 시장 데이터를 LLM 파인튜닝에 적합한 형식으로 변환하는 과정을 학습
- 다양한 파인튜닝 방법론에 맞는 데이터셋을 구성하는 방법 처리

### 2.1 데이터 수집

In [ ]:
import pandas as pd

# CSV 파일에서 로드
df = pd.read_csv('data/etf_info.csv', encoding='cp949')

# 데이터 확인
print(f"데이터 크기: {df.shape}")
print(f"컬럼: {df.columns.tolist()}")
df.head()

### 2.2 데이터 전처리

In [ ]:
# 한글 컬럼명 영문으로 매핑
column_mapping = {
    '표준코드': 'standard_code',
    '단축코드': 'ticker',
    '한글종목명': 'name_kr',
    '한글종목약명': 'short_name_kr',
    '영문종목명': 'name_en',
    '상장일': 'listing_date',
    '기초지수명': 'base_index',
    '지수산출기관': 'index_provider',
    '추적배수': 'tracking_multiplier',
    '복제방법': 'replication_method',
    '기초시장분류': 'base_market_category',
    '기초자산분류': 'base_asset_category',
    '상장좌수': 'listed_shares',
    '운용사': 'manager',
    'CU수량': 'cu_quantity',
    '총보수': 'total_expense_ratio',
    '과세유형': 'tax_type'
}

# 데이터프레임의 컬럼명 변경
df = df.rename(columns=column_mapping)

df.head()

### 2.3 Train/Test 분할

In [ ]:
from sklearn.model_selection import train_test_split

# 80/20 분할
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=1207,
    stratify=df['tax_type']  # 과세 유형별 균등 분할
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# 저장
train_df.to_csv('data/etf_train.csv', index=False)
test_df.to_csv('data/etf_test.csv', index=False)

### 2.4 기본 데이터셋 생성

`(1) Alpaca 형식 (LLM용)`

In [ ]:
from datasets import Dataset

def create_alpaca_dataset(df):
    """기본 Alpaca 형식 데이터셋"""
    alpaca_data = []
    
    for _, row in df.iterrows():
        # 패턴 1: 기본 정보
        alpaca_data.append({
            "instruction": "다음 ETF의 기본 정보를 제공해주세요.",
            "input": f"{row['name_kr']} (종목코드: {row['ticker']})",
            "output": f"{row['name_kr']}은 {row['manager']}에서 운용하는 ETF입니다. "
                     f"기초지수는 {row['base_index']}이며, "
                     f"총보수는 연 {row['total_expense_ratio']:.2f}%입니다. "
                     f"{pd.to_datetime(row['listing_date']).strftime('%Y년 %m월')}에 상장되었습니다."
        })
        
        # 패턴 2: 투자 특징
        alpaca_data.append({
            "instruction": "이 ETF의 특징을 알려주세요.",
            "input": f"{row['name_kr']}",
            "output": f"{row['name_kr']}는 {row['base_index']}를 추종하는 ETF로, "
                     f"{row['replication_method']} 방식으로 운용됩니다. "
                     f"기초자산은 {row['base_asset_category']}이며, "
                     f"과세 유형은 {row['tax_type']}입니다."
        })
        
        # 패턴 3: 운용사 질문
        alpaca_data.append({
            "instruction": "이 ETF의 운용사 정보를 알려주세요.",
            "input": f"{row['name_kr']}",
            "output": f"{row['name_kr']}는 {row['manager']}에서 운용하고 있습니다. "
                     f"총보수는 연 {row['total_expense_ratio']:.2f}%입니다."
        })
        
        # 패턴 4: 투자 적합성 (자산 분류별)
        suitability = {
            '주식': '시장 상승기에 유리하며, 변동성이 높아 리스크 관리가 필요합니다.',
            '채권': '안정적인 수익을 추구하며, 금리 변동에 영향을 받습니다.',
            '원자재': '인플레이션 헤지 수단으로 활용 가능하며, 시장 변동성이 큽니다.',
        }
        
        advice = suitability.get(
            row['base_asset_category'], 
            '해당 자산군의 특성을 충분히 이해하고 투자하시기 바랍니다.'
        )
        
        alpaca_data.append({
            "instruction": "이 ETF는 어떤 투자자에게 적합한가요?",
            "input": f"{row['name_kr']}",
            "output": f"{row['name_kr']}는 {row['base_asset_category']} 자산에 투자하는 ETF입니다. "
                     f"{advice}"
        })
    
    return Dataset.from_list(alpaca_data)

# 데이터셋 생성
train_alpaca = create_alpaca_dataset(train_df)
test_alpaca = create_alpaca_dataset(test_df)

print(f"기본 Alpaca 데이터: Train {len(train_alpaca)}, Test {len(test_alpaca)}")

In [ ]:
train_alpaca

In [ ]:
test_alpaca

In [ ]:
train_alpaca[0]

In [ ]:
test_alpaca[0]

In [ ]:
# 데이터셋 저장
output_dir = "datasets"
os.makedirs(output_dir, exist_ok=True)

train_alpaca.save_to_disk(f"{output_dir}/train_alpaca")
test_alpaca.save_to_disk(f"{output_dir}/test_alpaca")

In [ ]:
from datasets import load_from_disk

# 저장된 데이터셋 불러오기
train_alpaca = load_from_disk(f"{output_dir}/train_alpaca")
test_alpaca = load_from_disk(f"{output_dir}/test_alpaca")

# 데이터셋 확인
print(f"Train: {len(train_alpaca)}")
print(f"Test: {len(test_alpaca)}")

In [ ]:
from datasets import DatasetDict

# Train과 Test를 분리해서 하나의 데이터셋으로 구성
llm_alpaca_dataset = DatasetDict({
    'train': train_alpaca,
    'test': test_alpaca
})

print(f"✅ 데이터셋 구성:")
print(f"   Train: {len(llm_alpaca_dataset['train'])}개")
print(f"   Test: {len(llm_alpaca_dataset['test'])}개")

# 사용 예시
print("\n학습용 샘플:")
print(llm_alpaca_dataset['train'][0])

print("\n평가용 샘플:")
print(llm_alpaca_dataset['test'][0])

In [ ]:
# 데이터셋 허깅페이스 업로드
llm_alpaca_dataset.push_to_hub(
    "redwiggler/etf-alpaca-llm-v1",  # 자신의 사용자명으로 변경
    private=True,  # 비공개 여부 선택
)

In [ ]:
# 데이터셋 허깅페이스 다운로드
from datasets import load_dataset

# 데이터셋 다운로드
train_alpaca = load_dataset("redwiggler/etf-alpaca-llm-v1", split="train")
test_alpaca = load_dataset("redwiggler/etf-alpaca-llm-v1", split="test")

# 데이터셋 확인
print(f"Train: {len(train_alpaca)}")
print(f"Test: {len(test_alpaca)}")


`(2) 임베딩용 Positive Pairs`

In [ ]:
def create_embedding_dataset(df):
    """Hard Negatives를 포함한 임베딩 데이터"""
    from itertools import combinations
    
    pairs = []
    
    # 1. 기본 Positive Pairs (위와 동일)
    for _, row in df.iterrows():
        pairs.append({
            "sentence1": f"{row['name_kr']}",
            "sentence2": f"{row['base_index']}를 추종하는 {row['manager']} 운용 ETF",
            "label": 1  # 유사함
        })
        
        pairs.append({
            "sentence1": f"종목코드 {row['ticker']}",
            "sentence2": f"{row['name_kr']} ETF",
            "label": 1
        })
    
    # 2. Hard Negatives: 같은 운용사지만 다른 ETF
    for manager in df['manager'].unique():
        manager_etfs = df[df['manager'] == manager]
        
        if len(manager_etfs) > 1:
            # 같은 운용사의 ETF 조합
            for (idx1, row1), (idx2, row2) in combinations(manager_etfs.iterrows(), 2):
                pairs.append({
                    "sentence1": f"{row1['name_kr']}",
                    "sentence2": f"{row2['name_kr']}",
                    "label": 0  # 유사하지 않음 (같은 운용사지만 다른 상품)
                })
    
    # 3. Hard Negatives: 같은 기초지수지만 다른 ETF
    for index in df['base_index'].unique():
        index_etfs = df[df['base_index'] == index]
        
        if len(index_etfs) > 1:
            for (idx1, row1), (idx2, row2) in combinations(index_etfs.iterrows(), 2):
                pairs.append({
                    "sentence1": f"종목코드 {row1['ticker']}",
                    "sentence2": f"{row2['name_kr']}",
                    "label": 0  # 다른 ETF
                })
    
    # 4. Easy Negatives: 완전히 다른 카테고리
    for _, row1 in df.iterrows():
        # 다른 자산군 선택
        diff_category = df[df['base_asset_category'] != row1['base_asset_category']]
        
        if len(diff_category) > 0:
            row2 = diff_category.sample(1).iloc[0]
            pairs.append({
                "sentence1": f"{row1['name_kr']}",
                "sentence2": f"{row2['name_kr']}",
                "label": 0
            })
    
    return Dataset.from_list(pairs)

# 데이터셋 생성
train_embedding = create_embedding_dataset(train_df)
test_embedding = create_embedding_dataset(test_df)

print(f"임베딩 데이터: Train {len(train_embedding)}, Test {len(test_embedding)}")

In [ ]:
train_embedding

In [ ]:
test_embedding

In [ ]:
train_embedding[0]

In [ ]:
test_embedding[-1]

In [ ]:
# 데이터셋 저장
output_dir = "datasets"
os.makedirs(output_dir, exist_ok=True)

train_embedding.save_to_disk(f"{output_dir}/train_embedding")
test_embedding.save_to_disk(f"{output_dir}/test_embedding")


In [ ]:
# 저장된 데이터셋 불러오기
train_embedding = load_from_disk(f"{output_dir}/train_embedding")
test_embedding = load_from_disk(f"{output_dir}/test_embedding")

# 데이터셋 확인
print(f"Train: {len(train_embedding)}")
print(f"Test: {len(test_embedding)}")

In [ ]:
from datasets import DatasetDict

# Train과 Test를 분리해서 하나의 데이터셋으로 구성
embedding_dataset = DatasetDict({
    'train': train_embedding,
    'test': test_embedding
})

# 데이터셋 허깅페이스 업로드
embedding_dataset.push_to_hub(
    "redwiggler/etf-embedding-v1",  # 자신의 사용자명으로 변경
    private=True,  # 비공개 여부 선택
)



In [ ]:
# 데이터셋 허깅페이스 다운로드
from datasets import load_dataset

# 데이터셋 다운로드
train_embedding = load_dataset("redwiggler/etf-embedding-v1", split="train")
test_embedding = load_dataset("redwiggler/etf-embedding-v1", split="test")

# 데이터셋 확인
print(f"Train: {len(train_embedding)}")
print(f"Test: {len(test_embedding)}")


---

## 3. 데이터 증강 기법

### 3.1 LLM을 활용한 고품질 데이터 생성

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List
import time

# Pydantic 모델 정의
class QnAPair(BaseModel):
    """질문-답변 쌍"""
    question: str = Field(description="투자자의 자연스러운 질문")
    answer: str = Field(description="전문가의 상세한 답변")

class ETFQnASet(BaseModel):
    """5가지 질문-답변 세트"""
    qna_pairs: List[QnAPair] = Field(description="질문-답변 목록")

# LLM 초기화
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.2
)

# 구조화된 출력
structured_llm = llm.with_structured_output(ETFQnASet)

# 프롬프트 템플릿
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 ETF 투자 전문가입니다. 
    제공된 ETF 정보를 바탕으로 투자자가 물어볼 수 있는 자연스럽고 다양한 질문 5가지와 
    각 질문에 대한 전문적이고 상세한 답변을 생성해주세요. 반드시 제시된 ETF 정보에 근거하여 답변해주세요.
    
    답변은 다음을 포함해야 합니다:
    - 정확한 데이터 기반 정보
    - 투자자가 이해하기 쉬운 설명
    - 주의사항 및 리스크 안내
    - 실용적인 조언"""),
    ("human", """
    ETF 정보:
    {etf_info}
    
    다음 5가지 카테고리의 질문과 답변을 생성하세요:
    1. 기본 정보 (이름, 종목코드, 상장일 등)
    2. 투자 전략 (기초지수, 추종 방식, 포트폴리오)
    3. 비용 및 수익 (총보수, 배당, 세금)
    4. 위험 관리 (변동성, 추적오차, 주의사항)
    5. 투자 적합성 (목표 수익률, 투자 기간, 투자자 유형)
    
    각 답변은 최소 3문장 이상으로 작성하세요.
    """)
])

def generate_qa_data(df, batch_size=10):
    """OpenAI API로 고품질 Q&A 생성"""
    enhanced_data = []
    total_batches = (len(df) + batch_size - 1) // batch_size
    
    for batch_idx in range(total_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(df))
        batch_df = df.iloc[start_idx:end_idx]
        
        print(f"\n배치 {batch_idx+1}/{total_batches} 처리 중...")
        
        # 배치 프롬프트 생성
        batch_prompts = []
        for _, row in batch_df.iterrows():
            etf_info = f"""
종목코드: {row['ticker']}
ETF 이름: {row['name_kr']} ({row['name_en']})
운용사: {row['manager']}
기초지수: {row['base_index']}
지수 산출: {row['index_provider']}
추적배수: {row['tracking_multiplier']}x
복제방법: {row['replication_method']}
기초시장: {row['base_market_category']}
기초자산: {row['base_asset_category']}
총보수: {row['total_expense_ratio']:.2f}%
과세유형: {row['tax_type']}
상장일: {row['listing_date']}
            """.strip()
            
            batch_prompts.append(qa_prompt.format(etf_info=etf_info))
        
        # 배치 처리
        try:
            start_time = time.time()
            batch_results = structured_llm.batch(batch_prompts)
            elapsed = time.time() - start_time
            
            print(f"✅ 배치 완료 (소요: {elapsed:.2f}초)")
            
            # 결과 저장
            for i, qa_set in enumerate(batch_results):
                etf_name = batch_df.iloc[i]['name_kr']
                print(f"   - {etf_name}: {len(qa_set.qna_pairs)}개 Q&A")
                
                for qa in qa_set.qna_pairs:
                    enhanced_data.append({
                        "instruction": "다음 ETF 관련 질문에 전문가답게 답변해주세요.",
                        "input": qa.question,
                        "output": qa.answer,
                        "source": "openai_generated",
                        "etf_ticker": batch_df.iloc[i]['ticker']
                    })
            
            # Rate limit 방지
            if batch_idx < total_batches - 1:
                time.sleep(1)
                
        except Exception as e:
            print(f"⚠️ 오류 발생: {e}")
            # 개별 처리로 전환
            for idx, row in batch_df.iterrows():
                try:
                    etf_info = f"종목코드: {row['ticker']}\\n..."
                    qa_set = structured_llm.invoke(qa_prompt.format(etf_info=etf_info))
                    
                    for qa in qa_set.qna_pairs:
                        enhanced_data.append({
                            "instruction": "다음 ETF 관련 질문에 전문가답게 답변해주세요.",
                            "input": qa.question,
                            "output": qa.answer,
                            "source": "openai_generated",
                            "etf_ticker": row['ticker']
                        })
                    time.sleep(2)
                except Exception as inner_e:
                    print(f"   ⚠️ {row['name_kr']} 건너뜀: {inner_e}")
    
    return Dataset.from_list(enhanced_data)

# 데이터 생성 (샘플만)
sample_df = df.sample(n=10, random_state=1207)  # 테스트용
enhanced_dataset = generate_qa_data(sample_df, batch_size=5)

print(f"OpenAI 생성 데이터: {len(enhanced_dataset)}개")
enhanced_dataset[0]

### 3.2 역번역(Back-Translation) 데이터 증강

- 설치: pip install -U deep-translator

In [ ]:
from deep_translator import GoogleTranslator
from datasets import Dataset

def back_translate_deep(text, intermediate_lang='en'):
    """
    deep-translator를 이용한 역번역
    한국어 → 영어 → 한국어
    """
    try:
        # 한국어 → 중간 언어
        translated = GoogleTranslator(source='ko', target=intermediate_lang).translate(text)
        
        # 중간 언어 → 한국어
        back_translated = GoogleTranslator(source=intermediate_lang, target='ko').translate(translated)
        
        return back_translated
    except Exception as e:
        print(f"역번역 오류: {e}")
        return text

def augment_by_back_translation(dataset, languages=['en', 'ja', 'zh-CN']):
    """여러 언어로 역번역하여 데이터 증강"""
    augmented_data = []
    
    print(f"역번역 데이터 증강 중... (언어: {languages})")
    
    for idx, example in enumerate(dataset):
        # 원본 추가
        augmented_data.append(example)
        
        # 각 언어로 역번역
        for lang in languages:
            try:
                aug_input = back_translate_deep(example['input'], lang)
                aug_output = back_translate_deep(example['output'], lang)
                
                # 너무 유사하면 스킵
                if aug_input == example['input']:
                    continue
                
                augmented_data.append({
                    'instruction': example['instruction'],
                    'input': aug_input,
                    'output': aug_output,
                    'source': f'back_translation_{lang}',
                    'original_idx': idx
                })
                
                # Rate limit 방지
                import time
                time.sleep(0.5)
                
            except Exception as e:
                print(f"   오류 ({idx}, {lang}): {e}")
                continue
        
        if (idx + 1) % 10 == 0:
            print(f"   진행: {idx+1}/{len(dataset)}")
    
    return Dataset.from_list(augmented_data)

# 역번역 증강 (샘플)
sample_dataset = train_alpaca.select(range(20))
augmented_dataset = augment_by_back_translation(
    sample_dataset, 
    languages=['en']  # 영어만 사용
)

print(f"\n✅ 역번역 완료!")
print(f"   원본: {len(sample_dataset)}개")
print(f"   증강 후: {len(augmented_dataset)}개")
print(f"   증강 비율: {len(augmented_dataset)/len(sample_dataset):.1f}x")

# 샘플 확인
print("\n원본 vs 역번역:")
print(f"원본: {sample_dataset[0]['input']}")
print(f"역번역: {augmented_dataset[1]['input']}")

### 3.3 Synthetic Data Generation (패러프레이징)

In [ ]:
def paraphrase_with_llm(text, num_variations=3):
    """LLM으로 패러프레이즈 생성"""
    
    prompt = f"""다음 텍스트를 {num_variations}가지 다른 방식으로 표현해주세요. 
의미는 동일하게 유지하되, 문장 구조와 단어 선택을 다양하게 해주세요.

원문: {text}

다음 형식으로 응답하세요:
1. [첫 번째 패러프레이즈]
2. [두 번째 패러프레이즈]
3. [세 번째 패러프레이즈]
"""
    
    try:
        response = llm.invoke(prompt)
        # 번호 제거 및 분리
        variations = []
        for line in response.content.strip().split('\n'):
            line = line.strip()
            if line and line[0].isdigit():
                # "1. " 부분 제거
                text = line.split('.', 1)[1].strip()
                variations.append(text)
        
        return variations[:num_variations]
    except Exception as e:
        print(f"패러프레이즈 오류: {e}")
        return [text]

def augment_by_paraphrasing(dataset, num_variations=2):
    """패러프레이징으로 데이터 증강"""
    augmented = []
    
    print(f"패러프레이징 데이터 증강 중...")
    
    for idx, example in enumerate(dataset):
        # 원본
        augmented.append(example)
        
        # 질문 패러프레이즈
        question_variations = paraphrase_with_llm(
            example['input'], 
            num_variations
        )
        
        for var in question_variations:
            if var != example['input']:
                augmented.append({
                    'instruction': example['instruction'],
                    'input': var,
                    'output': example['output'],
                    'source': 'paraphrase',
                    'original_idx': idx
                })
        
        if (idx + 1) % 5 == 0:
            print(f"   진행: {idx+1}/{len(dataset)}")
            time.sleep(1)  # Rate limit
    
    return Dataset.from_list(augmented)

# 패러프레이징 (소량 샘플)
sample_for_para = train_alpaca.select(range(5))
paraphrased_dataset = augment_by_paraphrasing(sample_for_para, num_variations=2)

print(f"\n원본: {len(sample_for_para)}")
print(f"증강 후: {len(paraphrased_dataset)}")

In [ ]:
paraphrased_dataset

In [ ]:
paraphrased_dataset[0]

In [ ]:
paraphrased_dataset[2]

### 3.4 DPO 데이터셋 생성

In [ ]:
from datasets import Dataset
import pandas as pd
import re

def parse_tracking_multiplier(value):
    """
    추적배수 데이터 형식에 맞춘 파싱
    
    입력:
    - '일반' → 1.0
    - '2X 레버리지' → 2.0
    - '2X 인버스' → -2.0
    - '1X 인버스' → -1.0
    - '1.5X 레버리지' → 1.5
    """
    if pd.isna(value):
        return 1.0
    
    value_str = str(value).strip()
    
    # '일반'인 경우
    if value_str == '일반':
        return 1.0
    
    # 인버스 체크
    is_inverse = '인버스' in value_str
    
    # 숫자 추출 (1X, 2X, 1.5X 등)
    numbers = re.findall(r'(\d+\.?\d*)X?', value_str)
    
    if numbers:
        multiplier = float(numbers[0])
        return -multiplier if is_inverse else multiplier
    
    # 기본값
    return 1.0

def get_etf_type_info(value):
    """
    ETF 타입 정보 반환
    
    Returns:
        dict: {
            'type': '일반' | '레버리지' | '인버스',
            'multiplier': float,
            'display_name': str (화면 표시용)
        }
    """
    multiplier = parse_tracking_multiplier(value)
    value_str = str(value).strip()
    
    if '인버스' in value_str:
        return {
            'type': '인버스',
            'multiplier': multiplier,
            'display_name': value_str,
            'abs_multiplier': abs(multiplier)
        }
    elif '레버리지' in value_str:
        return {
            'type': '레버리지',
            'multiplier': multiplier,
            'display_name': value_str,
            'abs_multiplier': multiplier
        }
    else:
        return {
            'type': '일반',
            'multiplier': 1.0,
            'display_name': '일반형',
            'abs_multiplier': 1.0
        }

def create_dpo_dataset_final(df):
    """
    DPO 데이터셋 생성
    """
    dpo_data = []
    
    print("DPO 데이터셋 생성 중...")
    
    for idx, row in df.iterrows():
        # ETF 타입 정보 추출
        etf_info = get_etf_type_info(row['tracking_multiplier'])
        etf_type = etf_info['type']
        multiplier = etf_info['multiplier']
        display_name = etf_info['display_name']
        abs_mult = etf_info['abs_multiplier']
        
        # ===== 패턴 1: 기본 설명 (모든 ETF) =====
        prompt1 = f"{row['name_kr']} ETF에 대해 설명해주세요."
        
        # 레버리지 특성 문구
        if etf_type == '인버스':
            leverage_desc = f"{display_name} 상품으로 기초지수와 반대 방향({abs_mult}배)으로 움직이며 높은 변동성을 보입니다"
        elif etf_type == '레버리지':
            leverage_desc = f"{display_name} 상품으로 기초지수 움직임의 {abs_mult}배 변동성을 보입니다"
        else:
            leverage_desc = "일반형 상품으로 기초지수를 안정적으로 추종합니다"
        
        chosen1 = f"""{row['name_kr']}(종목코드: {row['ticker']})는 {row['manager']}에서 운용하는 ETF입니다.

**기본 정보**
- 기초지수: {row['base_index']}
- 추적 방식: {display_name}, {row['replication_method']} 복제
- 상장일: {row['listing_date']}

**비용 및 과세**
- 총보수: 연 {row['total_expense_ratio']:.2f}%
- 과세 유형: {row['tax_type']}

**투자 특성**
- 기초 시장: {row['base_market_category']}
- 기초 자산: {row['base_asset_category']}

이 ETF는 {leverage_desc}."""
        
        rejected1 = f"""{row['name_kr']}는 ETF입니다. 종목코드는 {row['ticker']}이고, {row['manager']}에서 만들었습니다. 
더 자세한 정보는 증권사 홈페이지를 참고하세요."""
        
        dpo_data.append({
            "prompt": prompt1,
            "chosen": chosen1,
            "rejected": rejected1,
            "etf_ticker": row['ticker'],
            "etf_type": etf_type
        })
        
        # ===== 패턴 2: 인버스 ETF 주의사항 =====
        if etf_type == '인버스':
            prompt2 = f"{row['name_kr']} ETF 투자 시 주의사항은 무엇인가요?"
            
            chosen2 = f"""{row['name_kr']}는 {display_name} ETF로, 다음 사항에 각별히 주의해야 합니다:

**1. 반대 방향 움직임**
기초지수({row['base_index']})가 상승하면 이 ETF는 하락하고, 지수가 하락하면 상승합니다. {abs_mult}배의 반대 방향 수익률을 추구합니다.

**2. 복리 효과의 함정**
일간 수익률을 추종하므로, 2일 이상 보유 시 복리 효과로 인해 기초지수의 누적 수익률과 큰 괴리가 발생할 수 있습니다.

**3. 적합한 투자 방식**
- ✅ 적합: 단기(1일) 하락장 대응, 헤지 목적
- ❌ 부적합: 장기 투자, 매수 후 보유 전략

**4. 높은 비용**
연 {row['total_expense_ratio']:.2f}%의 총보수가 발생하며, 이는 장기 보유 시 수익률에 큰 영향을 미칩니다.

**5. 변동성 위험**
{abs_mult}배의 변동성으로 인해 하루에도 큰 폭의 등락이 발생할 수 있습니다.

시장 방향성을 정확히 예측할 수 있고, 단기 트레이딩 경험이 있는 투자자만 신중히 투자하시기 바랍니다."""
            
            rejected2 = f"""{row['name_kr']}는 인버스 ETF라서 위험합니다. 초보자는 절대 투자하지 마세요. 손실 나기 딱 좋습니다."""
            
            dpo_data.append({
                "prompt": prompt2,
                "chosen": chosen2,
                "rejected": rejected2,
                "etf_ticker": row['ticker'],
                "etf_type": etf_type
            })
        
        # ===== 패턴 3: 레버리지 ETF 주의사항 =====
        elif etf_type == '레버리지':
            prompt3 = f"{row['name_kr']} ETF는 어떤 투자자에게 적합한가요?"
            
            chosen3 = f"""{row['name_kr']}는 {display_name} ETF로, 다음과 같은 투자자에게 적합합니다:

**적합한 투자자**
- 단기(1일~수일) 트레이딩 경험이 있는 투자자
- 시장 방향성에 대한 확신이 있는 투자자
- 높은 위험을 감수할 수 있는 투자자
- 적극적으로 포지션을 관리할 수 있는 투자자

**주의사항**
1. **높은 변동성**: 기초지수 변동의 {abs_mult}배 영향을 받아 수익과 손실이 모두 {abs_mult}배로 확대됩니다.

2. **복리 효과**: 일간 수익률 {abs_mult}배를 추종하므로, 장기 보유 시 기초지수 누적 수익률과 괴리가 발생합니다.

3. **적절한 보유 기간**: 1일~수일 정도의 단기 트레이딩용이며, 장기 투자에는 절대 부적합합니다.

4. **손절 필수**: 예상과 다른 방향으로 시장이 움직일 경우 즉시 손절해야 합니다.

5. **비용**: 연 {row['total_expense_ratio']:.2f}%의 높은 총보수가 발생합니다.

초보 투자자나 장기 투자자에게는 일반형 ETF를 권장합니다."""
            
            rejected3 = f"""{row['name_kr']}는 레버리지 ETF입니다. 수익률이 {abs_mult}배라서 좋아 보이지만 사실 위험합니다. 투자하지 마세요."""
            
            dpo_data.append({
                "prompt": prompt3,
                "chosen": chosen3,
                "rejected": rejected3,
                "etf_ticker": row['ticker'],
                "etf_type": etf_type
            })
        
        # ===== 패턴 4: 일반형 ETF 장점 =====
        else:  # 일반형
            prompt4 = f"{row['name_kr']} ETF의 장점은 무엇인가요?"
            
            chosen4 = f"""{row['name_kr']}는 일반형 ETF로, 다음과 같은 장점이 있습니다:

**1. 안정적인 추종**
{row['base_index']}를 1:1로 추종하여 기초지수와 유사한 수익률을 제공합니다. 복리 효과로 인한 괴리가 거의 없습니다.

**2. 합리적인 비용**
연 {row['total_expense_ratio']:.2f}%의 총보수로, 장기 투자 시에도 비용 부담이 적습니다.

**3. 장기 투자 적합**
레버리지나 인버스 상품과 달리 장기 보유가 가능하며, 중장기 자산 배분 전략에 적합합니다.

**4. 안정적인 운용**
{row['manager']}의 {row['replication_method']} 방식으로 안정적으로 운용됩니다.

**5. 다양한 활용**
- 장기 자산 배분의 핵심 자산
- 적립식 투자에 적합
- 포트폴리오의 안정적인 기반

{row['base_asset_category']} 자산에 장기적으로 투자하고자 하는 투자자에게 적합한 상품입니다."""
            
            rejected4 = f"""{row['name_kr']}는 그냥 평범한 ETF입니다. 레버리지 같은 특별한 기능도 없고, 수익률도 평범합니다."""
            
            dpo_data.append({
                "prompt": prompt4,
                "chosen": chosen4,
                "rejected": rejected4,
                "etf_ticker": row['ticker'],
                "etf_type": etf_type
            })
        
        # ===== 패턴 5: 비교 질문 (모든 ETF) =====
        prompt5 = f"{row['name_kr']}의 총보수는 경쟁력이 있나요?"
        
        # 평균 대비 비교 (실제로는 같은 카테고리 평균과 비교해야 하지만, 여기서는 단순화)
        if row['total_expense_ratio'] < 0.3:
            cost_eval = "매우 낮은 편"
        elif row['total_expense_ratio'] < 0.5:
            cost_eval = "합리적인 수준"
        else:
            cost_eval = "다소 높은 편"
        
        chosen5 = f"""{row['name_kr']}의 총보수는 연 {row['total_expense_ratio']:.2f}%로 {cost_eval}입니다.

**비용 분석**
- 총보수: {row['total_expense_ratio']:.2f}%
- ETF 유형: {display_name}
- 운용사: {row['manager']}

{etf_type} ETF는 일반형 ETF보다 운용 복잡도가 높아 보수가 다소 높은 경향이 있습니다. 
투자 결정 시 총보수뿐만 아니라 추적오차, 거래량, 운용사 신뢰도 등을 종합적으로 고려하시기 바랍니다.""" if etf_type != '일반' else f"""{row['name_kr']}의 총보수는 연 {row['total_expense_ratio']:.2f}%로 {cost_eval}입니다.

**비용 분석**
- 총보수: {row['total_expense_ratio']:.2f}%
- ETF 유형: 일반형
- 운용사: {row['manager']}

일반형 ETF는 비교적 낮은 비용으로 운용되며, 장기 투자 시 비용 차이가 수익률에 큰 영향을 미칩니다.
같은 지수를 추종하는 다른 ETF들과 비교하여 선택하시는 것을 권장합니다."""
        
        rejected5 = f"총보수는 {row['total_expense_ratio']:.2f}%입니다. 비싼지 싼지는 잘 모르겠네요."
        
        dpo_data.append({
            "prompt": prompt5,
            "chosen": chosen5,
            "rejected": rejected5,
            "etf_ticker": row['ticker'],
            "etf_type": etf_type
        })
        
        if (idx + 1) % 50 == 0:
            print(f"   진행: {idx+1}/{len(df)}")
    
    print(f"   완료: {len(df)}/{len(df)}")
    
    return Dataset.from_list(dpo_data)


In [ ]:
# 3DPO 데이터셋 생성
dpo_dataset = create_dpo_dataset_final(df)
dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=1207)

print(f"총 샘플: {len(dpo_dataset)}개")
print(f"Train: {len(dpo_split['train'])}개")
print(f"Test: {len(dpo_split['test'])}개")


In [ ]:
# ETF 타입별 분포
etf_types = [x['etf_type'] for x in dpo_dataset]
for etf_type in ['일반', '레버리지', '인버스']:
    count = etf_types.count(etf_type)
    if count > 0:
        print(f"   {etf_type}: {count}개 ({count/len(etf_types)*100:.1f}%)")


In [ ]:
dpo_split['train'][0]